# Mask R-CNN — Mask R-CNN (2017)

_Papers · Computer Vision_

**Add a small per-region mask network to Faster R-CNN, and fix its pixel misalignment with RoIAlign.**

---

This notebook is a practice scaffold for the **Mask R-CNN — Mask R-CNN (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)

# --- 0. Worked example: bilinear sample of a 4x4 map at the un-rounded point (y=2.7, x=1.3). ---
Fm = torch.tensor([[ 1.,  2.,  3.,  4.],
                   [ 5.,  6.,  7.,  8.],
                   [ 9., 10., 11., 12.],
                   [13., 14., 15., 16.]])

def bilinear(M, y, x):
    H, W = M.shape
    x0 = int(np.floor(x)); y0 = int(np.floor(y))
    x0 = max(0, min(W - 2, x0)); y0 = max(0, min(H - 2, y0))
    x1, y1 = x0 + 1, y0 + 1
    dx, dy = x - x0, y - y0                       # fractional offsets (NO rounding of x,y)
    return (M[y0, x0] * (1 - dx) * (1 - dy) + M[y0, x1] * dx * (1 - dy) +
            M[y1, x0] * (1 - dx) * dy       + M[y1, x1] * dx * dy)

by_hand = bilinear(Fm, 2.7, 1.3)
print("bilinear(2.7, 1.3) by hand =", round(float(by_hand), 4))   # 13.1

# Verify against PyTorch's bilinear sampler. grid_sample wants normalized coords in [-1,1].
def norm(p, N): return 2 * p / (N - 1) - 1
grid = torch.tensor([[[[norm(1.3, 4), norm(2.7, 4)]]]])           # (x, y) order, align_corners=True
gs = F.grid_sample(Fm.view(1, 1, 4, 4), grid, mode="bilinear", align_corners=True)
print("grid_sample             =", round(gs.item(), 4))           # 13.1  -> matches
print("RoIPool would read floor (2,1) =", float(Fm[2, 1]))        # 10.0  -> the misalignment


# --- 1. RoIAlign vs RoIPool on a toy 8x8 ramp feature map. ---
H = W = 8
yy, xx = torch.meshgrid(torch.arange(W).float(), torch.arange(H).float(), indexing="ij")
feat = (xx + yy)                                                  # smooth ramp, easy to read

def roi_align(M, roi, out):
    x1, y1, x2, y2 = roi
    bw, bh = (x2 - x1) / out, (y2 - y1) / out
    o = torch.zeros(out, out)
    for i in range(out):
        for j in range(out):
            cy = y1 + bh * (i + 0.5)                              # bin center, fractional
            cx = x1 + bw * (j + 0.5)
            o[i, j] = bilinear(M, cy, cx)                         # NO rounding
    return o

def roi_pool(M, roi, out):
    x1, y1, x2, y2 = [int(np.floor(v)) for v in roi]             # rounding #1: floor RoI
    bw, bh = (x2 - x1) / out, (y2 - y1) / out
    o = torch.zeros(out, out)
    for i in range(out):
        for j in range(out):
            ys = int(np.floor(y1 + i * bh)); ye = max(ys + 1, int(np.ceil(y1 + (i + 1) * bh)))
            xs = int(np.floor(x1 + j * bw)); xe = max(xs + 1, int(np.ceil(x1 + (j + 1) * bw)))
            o[i, j] = M[ys:ye, xs:xe].max()                      # rounding #2: integer bins
    return o

roi = (1.5, 1.5, 6.5, 5.5)                                       # NOT grid-aligned
print("\nRoIAlign 2x2:\n", roi_align(feat, roi, 2))
print("RoIPool  2x2:\n", roi_pool(feat, roi, 2))


# --- 2. ABLATION: slide the RoI right by sub-pixel steps; read one output bin from each. ---
base = (1.5, 1.5, 5.5, 5.5)
shifts = [round(0.1 * k, 1) for k in range(11)]
a_vals, p_vals = [], []
for s in shifts:
    r = (base[0] + s, base[1], base[2] + s, base[3])
    a_vals.append(round(float(roi_align(feat, r, 2)[0, 0]), 3))
    p_vals.append(round(float(roi_pool(feat, r, 2)[0, 0]), 3))
print("\nshift   :", shifts)
print("RoIAlign:", a_vals)    # smooth ramp 5.0 -> 6.0
print("RoIPool :", p_vals)    # staircase: stuck at 4.0, then jumps to 5.0
# RoIAlign tracks the sub-pixel shift; RoIPool quantizes it away -> the misalignment Mask R-CNN fixes.
# (Numbers from this toy run, not the paper's reported COCO AP.)

## Visualize the data & results

_As an RoI slides by sub-pixel amounts, does the cropped feature track the shift (RoIAlign) or snap in steps (RoIPool)?_

In [ ]:
import torch, numpy as np

# Toy 8x8 ramp feature map; one RoI slid right in sub-pixel steps. Reproduces the
# qualitative alignment effect: RoIAlign tracks the shift, RoIPool quantizes it away.
H = W = 8
yy, xx = torch.meshgrid(torch.arange(W).float(), torch.arange(H).float(), indexing="ij")
feat = xx + yy

def bilinear(M, y, x):
    x0 = max(0, min(W - 2, int(np.floor(x)))); y0 = max(0, min(H - 2, int(np.floor(y))))
    dx, dy = x - x0, y - y0
    return (M[y0, x0]*(1-dx)*(1-dy) + M[y0, x0+1]*dx*(1-dy) +
            M[y0+1, x0]*(1-dx)*dy   + M[y0+1, x0+1]*dx*dy)

def align_bin(roi):                                   # top-left bin center, no rounding
    x1, y1, x2, y2 = roi
    return float(bilinear(feat, y1 + (y2 - y1) / 4, x1 + (x2 - x1) / 4))

def pool_bin(roi):                                    # floor RoI, max over integer cells
    x1, y1, x2, y2 = [int(np.floor(v)) for v in roi]
    bw, bh = (x2 - x1) / 2, (y2 - y1) / 2
    ys, ye = int(np.floor(y1)), max(int(np.floor(y1)) + 1, int(np.ceil(y1 + bh)))
    xs, xe = int(np.floor(x1)), max(int(np.floor(x1)) + 1, int(np.ceil(x1 + bw)))
    return float(feat[ys:ye, xs:xe].max())

base = (1.5, 1.5, 5.5, 5.5)
shifts = [round(0.1 * k, 1) for k in range(11)]
print("shift   :", shifts)
print("RoIAlign:", [round(align_bin((base[0]+s, base[1], base[2]+s, base[3])), 3) for s in shifts])
print("RoIPool :", [round(pool_bin((base[0]+s, base[1], base[2]+s, base[3])), 3) for s in shifts])
# RoIAlign -> 5.0..6.0 smooth.  RoIPool -> 4.0 (x6) then 5.0 (staircase).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The alignment ablation. You have working roi_align and roi_pool.
            Slide one RoI right in steps of $0.1$ feature-cell and read the top-left output bin from each.
            What shape does each curve take, and what does the contrast prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hold the RoI's size fixed and add a sub-pixel offset $s \in \{0, 0.1, \dots, 1.0\}$ to its $x$-corners; recompute both crops at each $s$. — _Changing only the sub-pixel position isolates the alignment behaviour &mdash; everything else is identical._
- Plot the top-left bin's value vs $s$. RoIAlign rises smoothly and almost linearly; RoIPool stays flat across part of a cell, then jumps by a whole step. — _Bilinear sampling is continuous in the coordinate, so its output ramps; RoIPool rounds the coordinate, so its output is a step function &mdash; flat, then a jump._
- Conclude that RoIPool discards sub-pixel position (the misalignment) while RoIAlign preserves it. — _The staircase means many distinct true positions collapse to the same crop &mdash; exactly the effect the paper says ruins masks._

**Answer:** RoIAlign traces a smooth, near-linear ramp; RoIPool traces a staircase &mdash;
                 it stays frozen for part of a cell, then jumps a full step. In our toy run the RoIAlign bin
                 climbs $5.0 \to 6.0$ in even $0.1$ increments as the shift grows, while the RoIPool bin sits
                 at $4.0$ for the first half-cell, then snaps to $5.0$ and holds. Since only the sub-pixel
                 position changed, this isolates RoIPool's rounding as the cause of misalignment &mdash; the
                 staircase collapses many true positions to one crop, which is fatal for pixel-accurate masks.
                 The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** Repeat the worked example at a different point. Using the same $4\times4$ matrix $F$,
            bilinearly sample at $(y,x)=(1.5,\,2.5)$. Then state what RoIPool would read.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Neighbours: $x_0=2,y_0=1$, so $(1,2),(1,3),(2,2),(2,3)$ give $v_{00}=F[1,2]=7$, $v_{01}=F[1,3]=8$, $v_{10}=F[2,2]=11$, $v_{11}=F[2,3]=12$; offsets $dx=0.5,dy=0.5$. — _$\lfloor 2.5\rfloor=2$, $\lfloor 1.5\rfloor=1$; the fractional parts are both $0.5$._
- Blend: top $=0.5\cdot7+0.5\cdot8=7.5$, bottom $=0.5\cdot11+0.5\cdot12=11.5$, then $0.5\cdot7.5+0.5\cdot11.5=9.5$. — _With $dx=dy=0.5$ all four weights equal $0.25$, so the sample is the plain average of the four neighbours: $(7+8+11+12)/4=9.5$._
- RoIPool rounds $(1.5,2.5)\to(1,2)$ and reads $F[1,2]=7$. — _Flooring both coordinates throws away the half-cell offset, landing on the top-left corner only._

**Answer:** Bilinear gives $F(1.5,2.5)=\mathbf{9.5}$ &mdash; the average of the four surrounding values
                 $7,8,11,12$, since the point sits exactly at the center of its cell ($dx=dy=0.5$, all weights
                 $0.25$). RoIPool would instead read the rounded corner $F[1,2]=\mathbf{7}$. The $2.5$ gap is
                 the misalignment, again caused purely by rounding.

</details>

**Problem 3.** The mask branch outputs a $K\times m\times m$ tensor. For an RoI whose ground-truth class is
            $k=3$, which slice does $L_{mask}$ use, and why is the per-class, sigmoid design better than a
            single softmax mask?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $L_{mask}$ is computed only on channel $k=3$ &mdash; the $3$-rd $m\times m$ mask &mdash; with per-pixel binary cross-entropy; the other $K-1$ mask channels get no gradient from this RoI. — _The paper: "$L_{mask}$ is only defined on the $k$-th mask (other mask outputs do not contribute to the loss)."_
- Each channel is an independent "inside this object?" map via sigmoid, so classes don't compete for a pixel. — _A softmax across classes would force the masks to sum to one per pixel, coupling mask shape to class scoring._
- The classification head separately picks the label; at test time you read that label's mask channel. — _Decoupling mask from class means a wrong-but-close class still yields a sensible shape, and mask learning is not tangled with classification._

**Answer:** $L_{mask}$ uses only the $k=3$ channel (per-pixel sigmoid + binary cross-entropy); the
                 other channels are ignored for that RoI. This decoupling &mdash; one independent mask per
                 class, the class chosen by the classification head &mdash; means masks don't compete across
                 classes the way a per-pixel softmax would force them to. The paper reports this design
                 improves mask quality over the softmax alternative.

</details>